# IC substitution at multi-soliton

**Purpose.** Single-soliton: 200 scattered points fully replaced the IC.
Saha predicted this would fail at multi-soliton because the IC encodes
discrete info (peak count, amplitudes, positions) sparse data can't constrain.

**What's new.** In addition to N=3/6/9 at n_data=400, we add a **sample-size
control at N=3**: n_data ∈ {400, 1000, 2000} with no IC. If error stays >70%
even at 2000 points, the conclusion is rock-solid. If it drops a lot, the
original runs were just data-starved.

**Approx runtime:** ~4.5 h on Kaggle T4.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

for p in ["/kaggle/input/kdv-core", "/kaggle/working", "."]:
    if (Path(p) / "kdv_core.py").exists():
        sys.path.insert(0, p)
        break
import kdv_core as K
print(f"device = {K.DEVICE}")

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT.mkdir(parents=True, exist_ok=True)
print(f"output dir = {OUT}")


In [ ]:
K.quick_sanity_check(seed=99, adam_iter=500)

### Experiment matrix

In [ ]:
SEED = 99
BASELINES_WITH_IC = {3: 0.1580, 6: 0.4935, 9: 0.9754}  # from scaling notebook

RUNS = [
    # (N, n_data, use_ic, label)
    (3,  400,  False, "N3_noIC_n400"),
    (6,  400,  False, "N6_noIC_n400"),
    (9,  400,  False, "N9_noIC_n400"),
    (3, 1000,  False, "N3_noIC_n1000"),   # sample-size control
    (3, 2000,  False, "N3_noIC_n2000"),   # sample-size control
]

print(f"{'label':<20} {'N':>3} {'n_data':>7} {'use_ic':>8}")
for N, nd, uic, lbl in RUNS:
    print(f"  {lbl:<20} {N:>3} {nd:>7,} {str(uic):>8}")
print("\nWith-IC baselines (from scaling notebook):", BASELINES_WITH_IC)


### Run all configurations

In [ ]:
results = []
for N, n_data, use_ic, label in RUNS:
    ckpt_path = OUT / f"checkpoint_{label}.pt"

    if ckpt_path.exists():
        print(f"\n[resume] {label} done.")
        ck = K.load_checkpoint(ckpt_path)
        hist = ck["history"]
        results.append(dict(label=label, N=N, n_data=n_data, use_ic=use_ic,
                            L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                            time_min=(hist["adam_time"]+hist["lbfgs_time"])/60))
        continue

    print("\n" + "=" * 60)
    print(f"{label}  |  N={N}  n_data={n_data}  use_ic={use_ic}")
    print("=" * 60)
    K.set_seed(SEED)
    cfg = K.make_config(N)
    batch = K.build_data(cfg, seed=SEED, n_data=n_data, use_ic=use_ic)
    m = K.PINN(width=50).to(K.DEVICE)
    hist = K.train(m, batch, adam_iter=15000, lbfgs_iter=2000)
    K.save_checkpoint(ckpt_path, m, hist, cfg,
                      extras=dict(label=label, N=N, n_data=n_data,
                                  use_ic=use_ic, seed=SEED))
    print(f"  saved {ckpt_path}")

    results.append(dict(label=label, N=N, n_data=n_data, use_ic=use_ic,
                        L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                        time_min=(hist["adam_time"]+hist["lbfgs_time"])/60))
    pd.DataFrame(results).to_csv(OUT / "ic_substitution_results.csv", index=False)

df = pd.DataFrame(results)
df["baseline_with_IC"] = df["N"].map(BASELINES_WITH_IC)
df["ratio"]            = df["L2_final"] / df["baseline_with_IC"]
print("\n--- Results ---")
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
df.to_csv(OUT / "ic_substitution_results.csv", index=False)


### Plot 1 — Main comparison at n_data=400

In [ ]:
df_main = df[df["n_data"] == 400].sort_values("N")
x = np.arange(len(df_main))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - 0.2, df_main["baseline_with_IC"], width=0.4,
       color="C0", label="with IC (scaling baseline)", edgecolor="k")
ax.bar(x + 0.2, df_main["L2_final"], width=0.4,
       color="C3", label="no IC (data only, 400 pts)", edgecolor="k")
for i, (_, r) in enumerate(df_main.iterrows()):
    ax.text(i-0.2, r["baseline_with_IC"]+0.3, f"{r['baseline_with_IC']:.2f}",
            ha="center", fontsize=8)
    ax.text(i+0.2, r["L2_final"]+0.3, f"{r['L2_final']:.1f}",
            ha="center", fontsize=8)
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels([f"N={int(n)}" for n in df_main["N"]])
ax.set_ylabel("L2 error (%)"); ax.set_title("IC substitution at multi-soliton (n_data=400)")
ax.legend(); ax.grid(alpha=0.3, axis="y", which="both")
fig.tight_layout()
fig.savefig(OUT / "ic_substitution_main.png", dpi=140, bbox_inches="tight")
plt.show()


### Plot 2 — Sample-size control at N=3

Does more data eventually rescue the no-IC case?
If yes → original conclusion was data-limited.
If no  → IC genuinely encodes info data can't replace (confirms Saha's prediction).

In [ ]:
df_n3 = df[df["N"] == 3].sort_values("n_data")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_n3["n_data"], df_n3["L2_final"], "o-",
        color="C3", lw=2, ms=10, label="no IC (PDE+BC+data only)")
ax.axhline(BASELINES_WITH_IC[3], ls="--", color="C0",
           label=f"with-IC baseline = {BASELINES_WITH_IC[3]:.3f}%")
for _, r in df_n3.iterrows():
    ax.annotate(f"{r['L2_final']:.2f}%", (r["n_data"], r["L2_final"]),
                textcoords="offset points", xytext=(8, 6), fontsize=9)
ax.set_yscale("log")
ax.set_xlabel("n_data (scattered measurements)")
ax.set_ylabel("L2 error (%)")
ax.set_title("N=3 sample-size control — can more data replace the IC?")
ax.legend(); ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(OUT / "ic_substitution_samplesize_N3.png", dpi=140, bbox_inches="tight")
plt.show()


### Verdict

In [ ]:
print("=" * 60)
print("VERDICT")
print("=" * 60)
n3_rows = df[df["N"] == 3].sort_values("n_data")
for _, r in n3_rows.iterrows():
    print(f"  N=3  n_data={int(r['n_data']):>5,}  no-IC L2 = {r['L2_final']:.2f}%")

n3_2000 = df[(df["N"]==3) & (df["n_data"]==2000)]["L2_final"].iloc[0]
n3_400  = df[(df["N"]==3) & (df["n_data"]==400)]["L2_final"].iloc[0]
drop_factor = n3_400 / n3_2000

if n3_2000 < 5.0:
    print("\n  -> n_data=2000 RESCUES N=3 no-IC.")
    print("     Conclusion: data-limited, not truly IC-limited.")
elif drop_factor > 3:
    print(f"\n  -> n_data=2000 helps ({drop_factor:.1f}x drop) but does NOT rescue.")
    print("     Conclusion: IC encodes info data partially recovers but can't fully replace.")
else:
    print(f"\n  -> n_data=2000 barely helps (only {drop_factor:.1f}x drop).")
    print("     Conclusion: IC genuinely encodes discrete info sparse data cannot constrain.")
    print("     Saha's prediction CONFIRMED.")

print("\nDone. Outputs in", OUT)
